In [2]:
import os
os.environ.pop("DATABRICKS_CLUSTER_ID", None)
os.environ["DATABRICKS_SERVERLESS_COMPUTE_ID"] = "auto"

from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()
spark.sql("USE CATALOG sokoni_retail")
base = "abfss://sokoni@sokoniadls.dfs.core.windows.net/bronze_layer/"

In [2]:
sales = spark.table("sokoni_retail.bronze.sales_ingested")

total = sales.count()
distinct_ids = sales.select("sale_id").distinct().count()

print("total rows:", total)
print("distinct sale_id:", distinct_ids)
print("duplicate rows:", total - distinct_ids)

total rows: 1000000
distinct sale_id: 1000000
duplicate rows: 0


In [3]:
from pyspark.sql import functions as F

sales = spark.table("sokoni_retail.bronze.sales_ingested")

nulls = sales.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in sales.columns
])
nulls.show(truncate=False, vertical=True)

-RECORD 0-----------------
 product_id     | 0       
 sale_id        | 0       
 customer_id    | 0       
 store_id       | 0       
 sale_date      | 0       
 quantity       | 0       
 payment_method | 0       
 unit_price_kes | 0       
 _rescued_data  | 1000000 
 _ingested_at   | 0       
 _source_file   | 0       



In [4]:
sales = spark.table("sokoni_retail.bronze.sales_ingested")
sales.select("_rescued_data").show(3, truncate=False)

+-------------+
|_rescued_data|
+-------------+
|NULL         |
|NULL         |
|NULL         |
+-------------+
only showing top 3 rows


In [5]:
sales.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- sale_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- unit_price_kes: double (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)



In [6]:
sales = spark.table("sokoni_retail.bronze.sales_ingested")
print("rows:", sales.count())
print("rows with rescued data:", sales.filter("_rescued_data IS NOT NULL").count())

rows: 1000000
rows with rescued data: 0


In [7]:
from pyspark.sql import functions as F

sales = spark.table("sokoni_retail.bronze.sales_ingested")

print("rows:", sales.count())
print("distinct sale_id:", sales.select("sale_id").distinct().count())
print("distinct customers:", sales.select("customer_id").distinct().count())
print("distinct products:", sales.select("product_id").distinct().count())
print("distinct stores:", sales.select("store_id").distinct().count())

sales.select(
    F.min("sale_date").alias("first_date"),
    F.max("sale_date").alias("last_date"),
    F.min("quantity").alias("min_qty"),
    F.max("quantity").alias("max_qty"),
    F.min("unit_price_kes").alias("min_price"),
    F.max("unit_price_kes").alias("max_price"),
).show(truncate=False)

sales.groupBy("payment_method").count().show()

rows: 1000000
distinct sale_id: 1000000
distinct customers: 70000
distinct products: 1000
distinct stores: 11
+----------+----------+-------+-------+---------+---------+
|first_date|last_date |min_qty|max_qty|min_price|max_price|
+----------+----------+-------+-------+---------+---------+
|2025-01-01|2026-10-31|1      |5      |20.0     |1440.0   |
+----------+----------+-------+-------+---------+---------+

+--------------+------+
|payment_method| count|
+--------------+------+
|          Card|333419|
|        M-Pesa|333658|
|          Cash|332923|
+--------------+------+



In [13]:
from pyspark.sql import functions as F

sales = spark.table("sokoni_retail.bronze.sales_ingested")
total = sales.count()

print("duplicate sale_id rows:", total - sales.select("sale_id").distinct().count())
print("null store_id:", sales.filter("store_id IS NULL").count())
print("null unit_price_kes:", sales.filter("unit_price_kes IS NULL").count())
print("quantity outside 1-5:", sales.filter("quantity < 1 OR quantity > 5").count())
print("non-standard payment_method:",
      sales.filter(~F.col("payment_method").isin("M-Pesa", "Cash", "Card")).count())

duplicate sale_id rows: 47409
null store_id: 40106
null unit_price_kes: 9932
quantity outside 1-5: 20081
non-standard payment_method: 30034


In [14]:
spark.table("sokoni_retail.bronze.sales_ingested").show(10, truncate=False)

+----------+---------+-----------+--------+----------+--------+--------------+--------------+-------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|product_id|sale_id  |customer_id|store_id|sale_date |quantity|payment_method|unit_price_kes|_rescued_data|_ingested_at           |_source_file                                                                                                                                            |
+----------+---------+-----------+--------+----------+--------+--------------+--------------+-------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|P0973     |T00625000|C033875    |NULL    |2025-04-14|3       |Card          |140.0         |NULL         |2026-11-20 11:34:08.516|abfss://sokoni

In [15]:
from pyspark.sql import functions as F

sales = spark.table("sokoni_retail.bronze.sales_ingested")
sales.filter("store_id IS NULL").show(3, truncate=False)
sales.filter("quantity < 1 OR quantity > 5").show(3, truncate=False)
sales.filter(~F.col("payment_method").isin("M-Pesa", "Cash", "Card")).show(3, truncate=False)

+----------+---------+-----------+--------+----------+--------+--------------+--------------+-------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|product_id|sale_id  |customer_id|store_id|sale_date |quantity|payment_method|unit_price_kes|_rescued_data|_ingested_at           |_source_file                                                                                                                                            |
+----------+---------+-----------+--------+----------+--------+--------------+--------------+-------------+-----------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------+
|P0973     |T00625000|C033875    |NULL    |2025-04-14|3       |Card          |140.0         |NULL         |2026-11-20 11:34:08.516|abfss://sokoni

In [16]:
from pyspark.sql import functions as F

sales = spark.table("sokoni_retail.bronze.sales_ingested")
sales.filter("store_id IS NULL").show(3, truncate=False)
sales.filter("quantity < 1 OR quantity > 5").show(3, truncate=False)
sales.filter(~F.col("payment_method").isin("M-Pesa", "Cash", "Card")).show(3, truncate=False)

/home/muchai/sokoni_data_platform/EDA_notebooks/.venv/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py:2179: UserWarning: Spark Connect Session expired on the server. Please generate a new session by detaching and reattaching the compute if in a Databricks notebook or job or by calling DatabricksSession.builder.getOrCreate() if using Databricks Connect.
  warnings.warn(
/home/muchai/sokoni_data_platform/EDA_notebooks/.venv/lib/python3.10/site-packages/pyspark/sql/connect/client/reattach.py:270: UserWarning: ReleaseExecute failed with exception: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.FAILED_PRECONDITION
	details = "BAD_REQUEST: session_id is no longer usable. Generate a new session_id by detaching and reattaching the compute and then try again [sessionId=5895d079-c9c2-4870-848a-ea144f5a8ac6, reason=INACTIVITY_TIMEOUT].  (requestId=1d19c999-3dc9-4c34-8dfb-2f88eed45abe)"
	debug_error_string = "FAILED_PRECONDITION:BAD_REQUEST: session_id is no l

SparkConnectGrpcException: () BAD_REQUEST: session_id is no longer usable. Generate a new session_id by detaching and reattaching the compute and then try again [sessionId=5895d079-c9c2-4870-848a-ea144f5a8ac6, reason=INACTIVITY_TIMEOUT].  (requestId=c8b40eb7-2584-4662-be62-33662be99bb1)